# Análise dos elementos de interesse no portal da Prefeitura de Caruaru


``` 
data de inicio: 2023-09-21
data de conclusão: 
``` 

## Obtendo dados de Obras públicas


## Páginas de interesse


|NOME|URL|IMAGEM EXEMPLO|
|----|---|-----------|
|Página inicial|https://caruaru.pe.gov.br/portal-da-transparencia/|<img src="../prefeitura_municipal/img/home_portal_transparencia_prefeitura_caruaru.png"></img>|
|Seção "Obras_Públicas"|https://caruaru.pe.gov.br/portal-da-transparencia/obras-publicas/|<img src="../prefeitura_municipal/img/secao_obras_publicas_prefeitura_caruaru.png"></img>|
|Página de buscas por Obras Públicas|https://caruaru.pe.gov.br/portal-da-transparencia/obras-publicas/|<img src="../prefeitura_municipal/img/pagina_obras_publicas_prefeitura_caruaru.png"></img>|
|Exemplo de Dados de Obras Públicas (conteúdo de um dos cards)|https://caruaru.pe.gov.br/obras/dispensa-003-2024-ct-014-2024-contratacao-direta-de-empresa-especializada-na-prestacao-dos-servicos-de-manejo-dos-residuos-solidos-urbanos-no-municipio-de-caruaru-pe/|<img src="../prefeitura_municipal/img/conteudo_detalhado_obras_prefeitura.png"></img>|


## Exploração dos elementos de interesse:

### Xpath para obter todos os links para dados de obras públicas

![xpath](../prefeitura_municipal/img/caminho_para_card_obras.png)

```
//section[@class="groupBox contrast"]/a[contains(@class, "box status")]/@href
```

In [1]:
import requests
from parsel import Selector

In [2]:
url = "https://caruaru.pe.gov.br/portal-da-transparencia/obras-publicas/"
response = requests.get(url)

In [3]:
response

<Response [200]>

In [10]:
html_text = response.text
html_selector = Selector(html_text)

urls_obras = html_selector.xpath('//section[@class="groupBox contrast"]/a[contains(@class, "box status")]/@href').getall()

len(urls_obras)


24

## Obtendo dados das Obras 

In [11]:
obras_page = requests.get(urls_obras[1])

In [25]:
obras_page_text = obras_page.text
obras_selector = Selector(obras_page_text)
# Obtem todas as categorias (chaves principais) do formulário
keys = obras_selector.xpath('//section[@class="description-obra"]//div[@class="group-title"]/text()').getall()

keys



['MODALIDADE | Nº DA LICITAÇÃO',
 'DESCRIÇÃO DA OBRA',
 'CONVÊNIO',
 'CONTRATADO',
 'CONTRATO',
 'ADITIVO',
 'DESPESAS DO EXERCÍCIO',
 'VALOR PAGO ACUMULADO',
 'SITUAÇÃO',
 'ETAPA DA OBRA',
 'PERCENTUAL CONCLUÍDO',
 'TODOS OS DOCUMENTOS']

In [26]:
# Obtem todas as subcategorias (chaves aninhadas) do formulário
nested_keys = obras_selector.xpath('//div[@class="row-cards"]/div[@class="row-card"]/div[@class="card-title"]/text()').getall()
# Associa as subcategorias com sua respectiva categoria 
nested_keys


['Nº',
 'Concedente',
 'CPF | CNPJ',
 'Razão Social',
 'Nº',
 'Data ínicio',
 'Prazo',
 'Valor Contratado (R$)',
 'Data Conclusão / Paralisação',
 'Prazo Aditado',
 'Valor Aditado Acumulado (R$)',
 'Valor Médio Acumulado (R$)',
 'Valor pago Acumulado no período (R$)',
 'Valor pago Acumulado no exercício (R$)']

In [42]:
# Obtendo os valores das chaves e chaves aninhadas
retrive_values = obras_selector.xpath('//section[@class="description-obra"]/div[@class="row-details"]//div[@class="card-info"]/text()[normalize-space()]').getall()

retrive_values

['\n                                    Pregão Eletrônico nº 041/2023                                \n                                ',
 'Contratação de empresa especializada na prestação de serviço de fornecimento e transporte de água bruta, com serviço de irrigação, através de caminhões pipa com capacidade mínima de 12.000 litros para fins de irrigação de praças, parques urbanos, jardins, canteiros e demais áreas verdes públicas do município de Caruaru.',
 '\n                                            -',
 '\n                                            -',
 '\n                                            39.357.688/0001-05',
 '\n                                            TARUANDA EMPREENDIMENTOS LTDA',
 '\n                                            040/2023',
 '\n                                            07/08/2023',
 '\n                                            12 MESES ',
 '\n                                            R$ 2.349.878,52',
 '-',
 '\n                           

In [44]:
import re

value_list = []

for value in retrive_values:
    formated_item = re.sub(r"\s+", " ", value).strip()
    value_list.append(formated_item)

value_list

['Pregão Eletrônico nº 041/2023',
 'Contratação de empresa especializada na prestação de serviço de fornecimento e transporte de água bruta, com serviço de irrigação, através de caminhões pipa com capacidade mínima de 12.000 litros para fins de irrigação de praças, parques urbanos, jardins, canteiros e demais áreas verdes públicas do município de Caruaru.',
 '-',
 '-',
 '39.357.688/0001-05',
 'TARUANDA EMPREENDIMENTOS LTDA',
 '040/2023',
 '07/08/2023',
 '12 MESES',
 'R$ 2.349.878,52',
 '-',
 '-',
 'R$ 2.349.878,52',
 '0,0',
 '0,0',
 '0,0',
 '0,0',
 'em Andamento',
 'Serviço contínuo',
 '0.0%']

In [45]:
# Restrutua do formulário
form_structure = {
    keys[0]: value_list[0],
    keys[1]: value_list[1],
    keys[2]: {
        nested_keys[0]: value_list[2],
        nested_keys[1]: value_list[3],
    },
    keys[3]:{
        nested_keys[2]: value_list[4],
        nested_keys[3]: value_list[5],
    },
    keys[4]: {
        nested_keys[4]: value_list[6],
        nested_keys[5]: value_list[7],
        nested_keys[6]: value_list[8],
        nested_keys[7]: value_list[9],
        nested_keys[8]: value_list[10],
    },
    keys[5]: {
        nested_keys[9]: value_list[11],
        nested_keys[10]:value_list[12],
    },
    keys[6]: {
        nested_keys[11]:value_list[13],
        nested_keys[12]:value_list[14],
        nested_keys[13]:value_list[15],
    },
    keys[7]:value_list[16],
    keys[8]:value_list[17],
    keys[9]:value_list[18],
    keys[10]:value_list[19],
}

form_structure

{'MODALIDADE | Nº DA LICITAÇÃO': 'Pregão Eletrônico nº 041/2023',
 'DESCRIÇÃO DA OBRA': 'Contratação de empresa especializada na prestação de serviço de fornecimento e transporte de água bruta, com serviço de irrigação, através de caminhões pipa com capacidade mínima de 12.000 litros para fins de irrigação de praças, parques urbanos, jardins, canteiros e demais áreas verdes públicas do município de Caruaru.',
 'CONVÊNIO': {'Nº': '-', 'Concedente': '-'},
 'CONTRATADO': {'CPF | CNPJ': '39.357.688/0001-05',
  'Razão Social': 'TARUANDA EMPREENDIMENTOS LTDA'},
 'CONTRATO': {'Nº': '040/2023',
  'Data ínicio': '07/08/2023',
  'Prazo': '12 MESES',
  'Valor Contratado (R$)': 'R$ 2.349.878,52',
  'Data Conclusão / Paralisação': '-'},
 'ADITIVO': {'Prazo Aditado': '-',
  'Valor Aditado Acumulado (R$)': 'R$ 2.349.878,52'},
 'DESPESAS DO EXERCÍCIO': {'Valor Médio Acumulado (R$)': '0,0',
  'Valor pago Acumulado no período (R$)': '0,0',
  'Valor pago Acumulado no exercício (R$)': '0,0'},
 'VALOR PAGO

## Obtendo os documentos referentes aos contratos e a localização da obra